In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import json

In [4]:
phenotype_manifest = pd.read_csv("phenotype_manifest.csv", usecols=[
    'description',
    'phenocode',
    'filename'
    ])

In [7]:
with open("icd10_indices_n100.json", "r") as f:
    icd_indices = json.load(f)

In [8]:
icd_phenotypes = phenotype_manifest.iloc[icd_indices]

In [9]:
exclude_phenotypes = []

In [10]:
with open("exclude_phenotypes.txt", "r") as file:
    for line in file:
        if line[0]=='#':
            continue
        exclude_phenotypes.append(int(line.strip().split()[0]))

In [12]:
expected_folders = []

for index, row in icd_phenotypes.iterrows():
    if index in exclude_phenotypes:
        print(f"Skipping phenotype {index}: {row['description']} because it is on the list of excluded phenotypes.")
        expected_folders.append(np.nan)
    else:
        expected_folders.append(os.path.splitext(os.path.basename(row['filename']))[0])

In [14]:
metadata_columns = [
    "index",
    "description",
    "dir_name",
    "num_snps_found",
    "num_coding_snps_found",
    "num_overlapping_snps",
    "num_overlapping_loci",
    "num_original_list",
    "num_original_coding_snps",
    "p_value_threshold",
    "percentage_loci_recovered",
    "ld_based_clumping",
    "r2_threshold",
    "kb_radius",
    "window_size"]
collected_metadata_df = pd.DataFrame(columns=metadata_columns)
collected_metadata_df['description'] = icd_phenotypes['description']
collected_metadata_df['index'] = icd_phenotypes.index
collected_metadata_df['dir_name'] = expected_folders

In [18]:
missing_indeces = []

for idx, row in collected_metadata_df.iterrows():
    folder_path = f"results_1_sd_runtime/{row['dir_name']}"
    if not Path(folder_path):
        raise ValueError(f"No phecode directories found in {folder_path} folder")
    metadata_path = f"{folder_path}/metadata_df.csv"
    try:
        # Read the small single-row CSV
        temp_df = pd.read_csv(metadata_path)
        
        # Now update the corresponding fields in df1
        for col in temp_df.columns:
            if col in collected_metadata_df.columns:
                collected_metadata_df.at[idx, col] = temp_df.iloc[0][col]
                
    except FileNotFoundError:
        print(f"File {metadata_path} not found, skipping.")
        missing_indeces.append((idx, row['description']))    

File results_1_sd_runtime/phecode-327.41-both_sexes.tsv/metadata_df.csv not found, skipping.
File results_1_sd_runtime/phecode-743.11-both_sexes.tsv/metadata_df.csv not found, skipping.


In [22]:
i = 1
for idx, row in icd_phenotypes.iterrows():
    if idx == 5824 or idx == 6629:
        print(i)
    i += 1

203
721


In [19]:
missing_indeces

[(5824, 'Organic or persistent insomnia'), (6629, 'Osteoporosis NOS')]

In [23]:
# excluded phenotypes
collected_metadata_df['num_snps_found'].isna().sum()

np.int64(2)

In [24]:
# timed out phenotypes
collected_metadata_df['dir_name'].isna().sum()

np.int64(0)

In [25]:
collected_metadata_df

,index,description,dir_name,num_snps_found,num_coding_snps_found,num_overlapping_snps,num_overlapping_loci,num_original_list,num_original_coding_snps,p_value_threshold,percentage_loci_recovered,ld_based_clumping,r2_threshold,kb_radius,window_size
5466,5466,Staphylococcus infections,phecode-041.1-both_sexes.tsv,0,0,0,0,1,0,0.0,1.0,True,0.2,500,NaN
5467,5467,Streptococcus infection,phecode-041.2-both_sexes.tsv,1,0,0,0,0,0,0.0,1.0,True,0.2,500,NaN
5470,5470,Herpes zoster with nervous system complications,phecode-053.1-both_sexes.tsv,0,0,0,0,1,0,0.0,1.0,True,0.2,500,NaN
5475,5475,Chronic hepatitis,phecode-070.4-both_sexes.tsv,14,7,4,21,21,2,0.0,1.0,True,0.2,500,NaN
5476,5476,Hepatitis NOS,phecode-070.9-both_sexes.tsv,1,1,1,1,1,1,0.0,1.0,True,0.2,500,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6686,6686,Pallor and flushing,phecode-782.6-both_sexes.tsv,0,0,0,0,0,0,0.0,1.0,True,0.2,500,NaN
6692,6692,Other abnormal blood chemistry,phecode-790.6-both_sexes.tsv,1,1,0,1,1,0,0.0,1.0,True,0.2,500,NaN
6697,6697,Cardiogenic shock,phecode-797.1-both_sexes.tsv,1,0,0,0,0,0,0.0,1.0,True,0.2,500,NaN
6699,6699,Chronic fatigue syndrome,phecode-798.1-both_sexes.tsv,0,0,0,0,0,0,0.0,1.0,True,0.2,500,NaN


In [ ]:
collected_metadata_df.to_csv("collected_metadata_pathological_phens.csv")